# Feature engineering: predictive maintenance

This notebook builds and evaluates engineered features for the predictive maintenance model, including rolling aggregations, interaction terms, and scaling strategies.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("../data/sensor_readings.csv")
print(f"Dataset shape: {df.shape}")
print(f"Features: {list(df.columns)}")
df.head()

## Existing rolling features

The data loader already computes three engineered features. Let us inspect them.

In [ ]:
existing_eng = ["rolling_mean_temp_24h", "rolling_std_vibration_24h", "temp_pressure_ratio"]
print("Existing engineered features:")
print(df[existing_eng].describe().round(3))

print(f"\nMissing values in engineered features:")
print(df[existing_eng].isnull().sum())

## Additional interaction features

We create new features that capture relationships between sensor channels.

In [ ]:
# Vibration * temperature interaction (both rise before failure)
df["vibration_x_temp"] = df["vibration"] * df["temperature"]

# Power per RPM (efficiency indicator)
df["power_per_rpm"] = (df["power_consumption"] / df["rpm"].clip(lower=1)).round(4)

# Vibration to pressure ratio (bearing degradation signal)
df["vibration_pressure_ratio"] = (df["vibration"] / df["pressure"].clip(lower=0.5)).round(4)

# Hours per maintenance event (overdue maintenance indicator)
df["hours_per_maintenance"] = (
    df["operating_hours"] / df["maintenance_history_count"].clip(lower=1)
).round(1)

new_features = ["vibration_x_temp", "power_per_rpm", "vibration_pressure_ratio", "hours_per_maintenance"]
print("New feature statistics:")
print(df[new_features].describe().round(3))

## Visualize new features by class

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flat

for i, feat in enumerate(new_features):
    for label, color, name in [(0, "#3B6FD4", "Normal"), (1, "#E8C230", "Pre-failure")]:
        subset = df[df["failure_within_7days"] == label]
        axes[i].hist(subset[feat], bins=40, alpha=0.6, label=name, color=color, edgecolor="black")
    axes[i].set_title(feat)
    axes[i].legend()

plt.suptitle("New engineered features by class", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Feature correlation with target

In [ ]:
all_features = [c for c in df.columns if c not in ("failure_within_7days", "machine_id")]
corr_with_target = df[all_features + ["failure_within_7days"]].corr()["failure_within_7days"].drop("failure_within_7days")
corr_with_target = corr_with_target.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#E8C230" if abs(v) > 0.15 else "#3B6FD4" for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors, edgecolor="black")
ax.set_title("Feature correlation with failure_within_7days")
ax.set_xlabel("Pearson correlation")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print("Features ranked by absolute correlation with target:")
print(corr_with_target.abs().sort_values(ascending=False).round(3))

## Feature scaling comparison

In [ ]:
feature_cols = [c for c in df.columns if c not in ("failure_within_7days", "machine_id")]
X = df[feature_cols].values.astype(float)

# Standard scaling
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X)

# MinMax scaling
scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(X)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Original
axes[0].boxplot([X[:, i] for i in range(min(8, X.shape[1]))],
                labels=feature_cols[:8])
axes[0].set_title("Original scale")
axes[0].tick_params(axis="x", rotation=90)

# Standard
axes[1].boxplot([X_std[:, i] for i in range(min(8, X.shape[1]))],
                labels=feature_cols[:8])
axes[1].set_title("StandardScaler")
axes[1].tick_params(axis="x", rotation=90)

# MinMax
axes[2].boxplot([X_mm[:, i] for i in range(min(8, X.shape[1]))],
                labels=feature_cols[:8])
axes[2].set_title("MinMaxScaler")
axes[2].tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

print("StandardScaler is used for Logistic Regression; tree models use raw features.")

## Outlier detection

In [ ]:
sensor_cols = ["temperature", "vibration", "pressure", "rpm", "power_consumption"]

print("Outlier analysis (values beyond 3 standard deviations):\n")
for col in sensor_cols:
    mean = df[col].mean()
    std = df[col].std()
    outliers = df[(df[col] < mean - 3 * std) | (df[col] > mean + 3 * std)]
    n_outliers = len(outliers)
    pct = n_outliers / len(df) * 100
    failure_pct = outliers["failure_within_7days"].mean() * 100 if n_outliers > 0 else 0
    print(f"  {col}: {n_outliers} outliers ({pct:.2f}%), {failure_pct:.1f}% are pre-failure")

print("\nOutliers in sensor readings may be genuine pre-failure signals, so we retain them.")

## Train-test split and class balance check

In [ ]:
feature_cols_final = [c for c in df.columns if c not in ("failure_within_7days", "machine_id")]
X = df[feature_cols_final].values.astype(float)
y = df["failure_within_7days"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples, failure rate: {y_train.mean():.4f}")
print(f"Test set:     {X_test.shape[0]} samples, failure rate: {y_test.mean():.4f}")
print(f"Features:     {X_train.shape[1]}")
print(f"\nFeature list ({len(feature_cols_final)}):")
for i, f in enumerate(feature_cols_final, 1):
    print(f"  {i:2d}. {f}")

## Feature importance baseline (mutual information)

In [ ]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_df = pd.DataFrame({
    "Feature": feature_cols_final,
    "MI score": mi_scores,
}).sort_values("MI score", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(mi_df["Feature"], mi_df["MI score"], color="#3B6FD4", edgecolor="black")
ax.set_title("Mutual information with failure target")
ax.set_xlabel("MI score (nats)")
plt.tight_layout()
plt.show()

print("Top 5 features by mutual information:")
print(mi_df.sort_values("MI score", ascending=False).head().to_string(index=False))

## Multi-collinearity check

In [ ]:
corr_matrix = pd.DataFrame(X_train, columns=feature_cols_final).corr().abs()
upper = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))

high_corr = []
for col in upper.columns:
    for idx in upper.index:
        if upper.loc[idx, col] > 0.8:
            high_corr.append((idx, col, upper.loc[idx, col]))

if high_corr:
    print("Highly correlated feature pairs (|r| > 0.8):")
    for f1, f2, r in sorted(high_corr, key=lambda x: -x[2]):
        print(f"  {f1} <-> {f2}: {r:.3f}")
else:
    print("No feature pairs with |r| > 0.8 found.")

print("\nTree-based models handle collinearity well, so we keep all features.")

## Summary

Feature engineering decisions:

1. **Retained all existing rolling features** -- rolling_mean_temp_24h, rolling_std_vibration_24h, and temp_pressure_ratio all show separation between classes
2. **Created four new interaction features** -- vibration_x_temp, power_per_rpm, vibration_pressure_ratio, hours_per_maintenance
3. **StandardScaler for linear models only** -- tree-based models receive unscaled features
4. **No outlier removal** -- extreme sensor values are informative pre-failure signals
5. **class_weight="balanced" and scale_pos_weight** handle the 12:1 class imbalance without SMOTE
6. **All 15 features passed to modeling** -- collinearity is managed by tree ensembles